In [13]:
import os
import pandas as pd

# load the annotation files
ANNOTATOR1_DIR = "../annotations/annotator1"
ANNOTATOR2_DIR = "../annotations/annotator2"
ANNOTATOR3_DIR = "../annotations/annotator3" # only used as tiebreaker for now
OUTPUT_DIR = "../annotations/gt"
COLUMN_INDEX = 4  # Column E, keep paraphrase T/F

os.makedirs(OUTPUT_DIR, exist_ok=True)

# match files by modification type + model
def get_file_map(folder, prefix):
    files = [f for f in os.listdir(folder) if f.endswith(".xlsx") and f.startswith(prefix)]
    return {f[len(prefix):]: os.path.join(folder, f) for f in files}

a1_files = get_file_map(ANNOTATOR1_DIR, "a1_")
a2_files = get_file_map(ANNOTATOR2_DIR, "a2_")
a3_files = get_file_map(ANNOTATOR3_DIR, "a3a_")

# TODO: need to add check for a3 as well
common_suffixes = sorted(set(a1_files.keys()) & set(a2_files.keys()))

if not common_suffixes:
    raise ValueError("No matching files found between annotator1 and annotator2")

In [7]:
print(common_suffixes)

['AAE_chatgpt.xlsx', 'AAE_deepseek.xlsx', 'change_voice_chatgpt.xlsx', 'change_voice_deepseek.xlsx', 'formal_chatgpt.xlsx', 'formal_deepseek.xlsx', 'prepositions_chatgpt.xlsx', 'prepositions_deepseek.xlsx', 'synonym_substitution_chatgpt.xlsx', 'synonym_substitution_deepseek.xlsx']


In [15]:
# make sure labels are the same
def normalize_labels(series):
    mapping = {
        "TRUE": True, "T": True, "1": True, "YES": True,
        "FALSE": False, "F": False, "0": False, "NO": False
    }
    return (
        series.astype(str)
        .str.strip()
        .str.upper()
        .map(mapping)   
        .infer_objects(copy=False)
    )

In [17]:
# generate ground truth annotation files for automatic filtering rules
for suffix in common_suffixes:
    df1 = pd.read_excel(a1_files[suffix])
    df2 = pd.read_excel(a2_files[suffix])
    df3 = pd.read_excel(a3_files[suffix])  # (tiebreaker)

    # normalize labels
    col1 = normalize_labels(df1.iloc[:, COLUMN_INDEX])
    col2 = normalize_labels(df2.iloc[:, COLUMN_INDEX])
    col3 = normalize_labels(df3.iloc[:, COLUMN_INDEX])  # note: may contain blanks

    col_name = df1.columns[COLUMN_INDEX]

    final_labels = []
    for v1, v2, v3 in zip(col1, col2, col3):
        if pd.isna(v1) and pd.isna(v2):
            final_labels.append("")  # both missing
        elif v1 == v2:
            final_labels.append("T" if v1 else "F")  # agreement
        else:
            # disagreement -> use annotator3
            if pd.notna(v3):
                final_labels.append("T" if v3 else "F")
            else: # fallback, but shouldn't be used
                final_labels.append("T" if v1 else "F")

    # update the column with adjudicated labels
    df1[col_name] = final_labels

    # clear specified columns
    for col in ["uncertain", "wrong_modif", "realism", "meaning"]:
        if col in df1.columns:
            df1[col] = ""

    output_path = os.path.join(OUTPUT_DIR, f"annotated_{suffix}")
    df1.to_excel(output_path, index=False)
    print(f"Processed {suffix} → {output_path}")

Processed AAE_chatgpt.xlsx → ../annotations/gt/annotated_AAE_chatgpt.xlsx
Processed AAE_deepseek.xlsx → ../annotations/gt/annotated_AAE_deepseek.xlsx
Processed change_voice_chatgpt.xlsx → ../annotations/gt/annotated_change_voice_chatgpt.xlsx
Processed change_voice_deepseek.xlsx → ../annotations/gt/annotated_change_voice_deepseek.xlsx
Processed formal_chatgpt.xlsx → ../annotations/gt/annotated_formal_chatgpt.xlsx
Processed formal_deepseek.xlsx → ../annotations/gt/annotated_formal_deepseek.xlsx
Processed prepositions_chatgpt.xlsx → ../annotations/gt/annotated_prepositions_chatgpt.xlsx
Processed prepositions_deepseek.xlsx → ../annotations/gt/annotated_prepositions_deepseek.xlsx
Processed synonym_substitution_chatgpt.xlsx → ../annotations/gt/annotated_synonym_substitution_chatgpt.xlsx
Processed synonym_substitution_deepseek.xlsx → ../annotations/gt/annotated_synonym_substitution_deepseek.xlsx
